# Parte 1 · Conocer los datos 🔍

**Qué hace este archivo:** abre las ~250.000 compras reales del BPI Challenge 2019 y responde 5 preguntas básicas antes de analizar nada.

**Cómo usarlo:**
1. Arriba del todo, pulsa **«Run All»** (▶▶). Si te pregunta qué *kernel* usar, elige **Python 3.12**.
2. Espera. **La primera vez tarda varios minutos** (lee un archivo enorme). Después lo guarda en un formato rápido y las siguientes veces tarda segundos.
3. Lee los resultados que salen debajo de cada bloque y cópiamelos (o hazme captura) para interpretarlos juntos.

> Recordatorio del proceso: **pedir → llega la mercancía → llega la factura → pagar.**

## 0 · Abrir los datos
Este bloque busca el archivo en la carpeta `datos` y lo abre.

In [1]:
from pathlib import Path
import time
import pandas as pd

CARPETA_DATOS = Path("datos")
GUARDADO_RAPIDO = CARPETA_DATOS / "bpic2019.parquet"

inicio = time.time()

if GUARDADO_RAPIDO.exists():
    print("Abriendo la copia rápida que se guardó la otra vez...")
    df = pd.read_parquet(GUARDADO_RAPIDO)
else:
    archivos = sorted(list(CARPETA_DATOS.glob("*.xes")) + list(CARPETA_DATOS.glob("*.xes.gz")))
    if not archivos:
        raise FileNotFoundError(
            "No encuentro el archivo .xes en la carpeta 'datos'. "
            "Descárgalo (mira datos/LEEME.md), descomprímelo si es .zip y mételo ahí."
        )
    import pm4py
    print(f"Leyendo {archivos[0].name} ... (la primera vez tarda varios minutos, paciencia ☕)")
    df = pm4py.read_xes(str(archivos[0]))

    # Pequeños arreglos para que se pueda guardar en formato rápido
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True).dt.tz_localize(None)
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string")

    df.to_parquet(GUARDADO_RAPIDO)
    print("Guardada una copia rápida en", GUARDADO_RAPIDO)

# Nombres de columnas más fáciles de leer
df = df.rename(columns={
    "case:concept:name": "compra",       # identificador de cada compra (línea de pedido)
    "concept:name": "actividad",         # el paso del proceso (p. ej. 'Record Invoice Receipt')
    "time:timestamp": "fecha",           # cuándo ocurrió ese paso
    "case:Item Category": "tipo_compra", # tipo de compra (ver pregunta 4)
    "org:resource": "quien",             # persona o sistema que hizo el paso
})

print(f"Listo en {time.time() - inicio:.0f} segundos. Tabla con {len(df):,} filas y {df.shape[1]} columnas.")
df.head()

Abriendo la copia rápida que se guardó la otra vez...
Listo en 1 segundos. Tabla con 1,595,923 filas y 21 columnas.


,User,quien,actividad,Cumulative net worth (EUR),fecha,case:Spend area text,case:Company,case:Document Type,case:Sub spend area text,case:Purchasing Document,...,case:Vendor,case:Item Type,tipo_compra,case:Spend classification text,case:Source,case:Name,case:GR-Based Inv. Verif.,case:Item,compra,case:Goods Receipt
0,batch_00,batch_00,SRM: Created,298.0,2018-01-02 12:53:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
1,batch_00,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
2,batch_00,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
3,batch_00,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
4,batch_00,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True


**Cómo leer la tabla de arriba:** cada **fila es un paso** de una compra (un *evento*). Varias filas con el mismo número de `compra` son los pasos de esa misma compra, uno detrás de otro.

## Pregunta 1 · ¿Cuántas compras y cuántos pasos hay?

In [2]:
n_compras = df["compra"].nunique()
n_eventos = len(df)
print(f"Compras distintas:   {n_compras:,}")
print(f"Pasos registrados:   {n_eventos:,}")
print(f"Media de pasos por compra: {n_eventos / n_compras:.1f}")

Compras distintas:   251,734
Pasos registrados:   1,595,923
Media de pasos por compra: 6.3


## Pregunta 2 · ¿Qué pasos tiene el proceso?
Lista de actividades, de la más frecuente a la menos. Fíjate en las que **no deberían existir en un proceso ideal** (cambios de precio, cambios de cantidad, cancelaciones...): ahí empieza el análisis de las siguientes partes.

In [3]:
print("Número de actividades distintas:", df["actividad"].nunique())
pasos = (
    df.groupby("actividad")
      .agg(veces=("actividad", "size"), compras=("compra", "nunique"))
      .sort_values("compras", ascending=False)
)
pasos["% de compras"] = (100 * pasos["compras"] / n_compras).round(1)
pasos.head(20)

Número de actividades distintas: 42


,veces,compras,% de compras
actividad,,,
Create Purchase Order Item,251734,251734,100.0
Record Goods Receipt,314097,234479,93.1
Record Invoice Receipt,228760,211379,84.0
Vendor creates invoice,219919,209946,83.4
Clear Invoice,194393,183677,73.0
Remove Payment Block,57136,55839,22.2
Create Purchase Requisition Item,46592,46592,18.5
Receive Order Confirmation,32065,32061,12.7
Change Quantity,21449,17590,7.0


## Pregunta 3 · ¿De qué fechas son los datos?
Los datos deberían ser de **2018** (y algo de inicio de 2019). Si aparecen años raros (1948, 2020...), son **errores de registro**: es normal en datos reales y lo apuntamos como hallazgo.

In [4]:
print("Primer paso registrado:", df["fecha"].min())
print("Último paso registrado:", df["fecha"].max())
print()
print("Pasos por año:")
print(df["fecha"].dt.year.value_counts().sort_index().to_string())

Primer paso registrado: 1948-01-26 22:59:00
Último paso registrado: 2020-04-09 21:59:00

Pasos por año:
fecha
1948         10
1993          9
2001         22
2008         45
2015          3
2016          6
2017        223
2018    1550468
2019      45135
2020          2


## Pregunta 4 · ¿Qué tipos de compra hay?
La empresa usa 4 maneras de comprobar las facturas:
- **3-way match, invoice before GR:** se compara *pedido + albarán (mercancía recibida) + factura*, y **la factura puede llegar antes que la mercancía**. Si llega antes, se **bloquea el pago** hasta que llegue la mercancía (de ahí la actividad *Remove Payment Block*).
- **3-way match, invoice after GR:** igual, pero la factura solo se registra **después** de recibir la mercancía.
- **2-way match:** sin albarán (p. ej. servicios); solo pedido + factura.
- **Consignment:** material en consigna; no hay factura en el pedido.

Mira cuál domina: ese es el proceso real de la empresa, y el que vamos a analizar.

In [5]:
tipos = df.drop_duplicates("compra")["tipo_compra"].value_counts()
tabla_tipos = pd.DataFrame({"compras": tipos, "% del total": (100 * tipos / tipos.sum()).round(1)})
tabla_tipos

,compras,% del total
tipo_compra,,
"3-way match, invoice before GR",221010,87.8
"3-way match, invoice after GR",15182,6.0
Consignment,14498,5.8
2-way match,1044,0.4


## Pregunta 5 · Quedarnos solo con lo que vamos a estudiar
Para que el análisis sea limpio, nos quedamos con:
- compras del tipo **«3-way match, invoice before GR»** (el 88 % del total), y
- que **empiezan y terminan entre el 1-ene-2018 y el 31-ene-2019** (fuera los años raros).

Lo guardamos aparte para usarlo en las partes 2, 3 y 4.

In [6]:
TIPO = "3-way match, invoice before GR"
DESDE, HASTA = pd.Timestamp("2018-01-01"), pd.Timestamp("2019-02-01")

sub = df[df["tipo_compra"] == TIPO]
limites = sub.groupby("compra")["fecha"].agg(["min", "max"])
compras_ok = limites[(limites["min"] >= DESDE) & (limites["max"] < HASTA)].index
sub = sub[sub["compra"].isin(compras_ok)].sort_values(["compra", "fecha"])

sub.to_parquet(CARPETA_DATOS / "bpic2019_3way_before_gr.parquet")
print(f"Compras que estudiaremos: {sub['compra'].nunique():,} "
      f"({100 * sub['compra'].nunique() / n_compras:.1f} % del total)")
print(f"Pasos de esas compras:    {len(sub):,}")

Compras que estudiaremos: 220,810 (87.7 % del total)
Pasos de esas compras:    1,233,410


## Extra · La misma pregunta, en SQL
En las empresas, los datos de SAP se consultan muchas veces con **SQL**. Esta es la lista de pasos de las compras que vamos a estudiar, escrita en SQL (con DuckDB). Es una buena frase para el CV: *«consultas SQL sobre un log de eventos de SAP»*.

In [7]:
import duckdb

duckdb.sql('''
    SELECT actividad,
           COUNT(*)               AS veces,
           COUNT(DISTINCT compra) AS compras
    FROM sub
    GROUP BY actividad
    ORDER BY compras DESC
    LIMIT 15
''').df()

,actividad,veces,compras
0,Create Purchase Order Item,220810,220810
1,Record Goods Receipt,222529,206276
2,Record Invoice Receipt,208891,199368
3,Vendor creates invoice,203152,198342
4,Clear Invoice,180156,173503
5,Remove Payment Block,54622,53590
6,Create Purchase Requisition Item,42564,42564
7,Receive Order Confirmation,31266,31262
8,Change Quantity,19104,15586
9,Change Price,11125,10066


## ✍️ Conclusiones de la Parte 1

- Los datos tienen  251.734 compras y 1.595.923 pasos registrados, de enero de 2018 a enero de 2019.
- El proceso tiene 42 actividades distintas; las más frecuentes son Create Purchase Order Item,
Record Goods Receipt, Record Invoice Receipt,
Vendor creates invoice.
- Hay actividades que indican retrabajo, como Change Quantity (7% de las compras), Change Price (4,5%), Remove Payment Block (22,2%).
- Encontré errores de fechas (años como 1948 o 2020), que he excluido.
- Me centro en «3-way match, invoice before GR», el tipo mayoritario: 220.810 compras (87,7 % del total).